In [ ]:
import math
import random
import pandas as pd
from datetime import timedelta
from collections import defaultdict
import csv
import ast
from datetime import datetime
# === CONFIGURATION ===

social_media_apps = [
    "com.facebook.katana",            # Facebook
    "com.instagram.android",          # Instagram
    "com.whatsapp",                   # WhatsApp
    "com.facebook.orca",              # Facebook Messenger
    "com.google.android.youtube",     # YouTube
    "com.snapchat.android",           # Snapchat
    "com.twitter.android",            # Twitter (now X)
    "com.reddit.frontpage",           # Reddit
    "com.pinterest",                  # Pinterest
    "com.tiktok.android",             # TikTok
    "com.linkedin.android",           # LinkedIn
    "org.telegram.messenger",         # Telegram
    "com.threads",                    # Threads (Meta)
    "com.signal.android",             # Signal
    "com.discord",                    # Discord
    "tv.twitch.android.app",          # Twitch
    "com.quora.android",              # Quora
    "com.imo.android.imoim",          # imo
    "com.viber.voip",                 # Viber
    "com.tumblr",                     # Tumblr
    "com.rumble.video",               # Rumble
    "com.triller.android",            # Triller (U.S. downloads notable) :contentReference[oaicite:0]{index=0}
    "app.clapper.social",             # Clapper :contentReference[oaicite:1]{index=1}
    "com.spotify.music",              # Spotify (social shared playlists)
    "com.vevo.android",               # Vevo
    "com.teamx.android",              # Microsoft Teams (used for messaging & communities) :contentReference[oaicite:2]{index=2}
    "com.linecorp.line",              # Line (used in U.S. ethnic communities) :contentReference[oaicite:3]{index=3}
    "com.bsky.app",                   # Bluesky (emerging U.S. alternative)
    "com.beatreal.android",           # BeReal :contentReference[oaicite:4]{index=4}
    "com.snapchat.android",           # (Duplicate kept for emphasis)
    "com.xiaohongshu.app",            # Xiaohongshu (growing in U.S.) :contentReference[oaicite:5]{index=5}
    "com.triller.android",            # (Duplicate—Triller already listed)
    "com.lemon8.android",             # Lemon8 (increasing traction in U.S.) :contentReference[oaicite:6]{index=6}
    "com.zigazoo.android",            # Zigazoo (U.S. growth due to TikTok ban concerns) :contentReference[oaicite:7]{index=7}
    "com.clapper.android",            # (Same as Clapper for consistency)
    "com.bumble.app",                 # Bumble (socializing via dating)
    "com.meetup",                     # Meetup (community-focused groups)
    "com.gab.android",                # Gab (some U.S. niche usage)
    "com.patreon.android",            # Patreon (creator communities)
    "com.sclub.community",           # Some niche U.S.-based social
]

test_type = 'A' # Change this to 'A', 'B', 'C', or 'D' as needed

session_division_seconds = 120 if test_type == 'A' or test_type == 'C' else 45

In [ ]:
#Learning Agent and reward functions

# === Q-TABLE AGENT ===
class QLearningAgent:
    def __init__(self):
        self.q_table = defaultdict(lambda: [0.0, 0.0])  # 2 actions: 0 (no vibrate), 1 (vibrate)
        self.alpha = 0.1
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_decay = 0.99
        self.epsilon_min = 0.1

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 1)
        return int(self.q_table[state][1] > self.q_table[state][0])

    def learn(self, state, next_state, action, reward):
        best_next = max(self.q_table[next_state])
        td_target = reward + self.gamma * best_next
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error
        self.epsilon = max(self.epsilon * self.epsilon_decay, self.epsilon_min)

# === REWARD FUNCTIONS ===
config = {
    'vibration_penalty': -40, #-50, -60
    'vibration_compliance_reward': 60,
    'session_break_reward': 40,   
    'long_session_penalty': -40
}

def reward_fn_vibration(action, session_end_time, next_session_start_time, median_target_app_usage, session, config):
    reward = 0.0
    reward += session['total_vibrations'] * config['vibration_penalty']
    reward += session['complied_vibrations'] * config['vibration_compliance_reward']
    if action == 0 and  len(list(set(session['app_ids']) & set(social_media_apps))) > 0 and session['duration'] > timedelta(minutes=median_target_app_usage, seconds=0):
            reward += config['long_session_penalty']
    if action == 1 and next_session_start_time:
        break_duration = (next_session_start_time - session_end_time).total_seconds() / 60.0
        if break_duration >= timedelta(minutes=2).total_seconds() / 60.0:
            reward += config['session_break_reward']
    return reward




In [15]:

short_session_length = 0

median_target_app_usage = 0

def is_short_session(duration):
    return duration.total_seconds() <= short_session_length

def get_day_quarter(timestamp):
    hour = timestamp.hour
    if 0 <= hour < 6:
        return 0  # Night
    elif 6 <= hour < 12:
        return 1  # Morning
    elif 12 <= hour < 18:
        return 2  # Afternoon
    else:
        return 3  # Evening
    
def get_day_of_week(timestamp):
    return timestamp.weekday() >= 0 and timestamp.weekday() < 5  # Monday=0, Sunday=6

def is_target_app(app_name):
    return app_name in social_media_apps


def extract_state(first_app, session):
    if session is None:
        return (0, 0, 0, 0)  # Default state for terminal state
    return (
        int(is_target_app(first_app)),
        int(is_short_session(session['duration'])),
        get_day_quarter(session['start_time']),
        int(get_day_of_week(session['start_time']))
    )
    
    
#write a function to calculate median of target app usage in app sessions
def calculate_median_target_app_usage(baseline_week):
    target_app_usages = []
    for day_sessions in baseline_week:
        total_target_app_time = sum((session['duration'] for session in day_sessions if session['app_name'] in social_media_apps), timedelta(0))
        target_app_usages.append(total_target_app_time)
    return math.floor(pd.Series(target_app_usages).median().total_seconds() / 60)

# calculate duration which is considered short session based on 50th percentile of session durations in baseline week
def calculate_short_session_length(baseline_week):
    session_durations = []
    for day_sessions in baseline_week:
        for session in day_sessions:
            session_durations.append(session['duration'])
    return pd.Series(session_durations).quantile(0.5)


def calculate_75_percentile_length(baseline_week):
    session_durations = []
    for day_sessions in baseline_week:
        for session in day_sessions:
            session_durations.append(session['duration'])
    return pd.Series(session_durations).quantile(0.75)

def get_first_app_in_group(day_sessions, group_id):
    for session in day_sessions:
        if session['group_id'] == group_id:
            return session['app_name']
    return None

def get_grouped_day_sessions(day_sessions):
    grouped_sessions = []
    current_group_sessions = []
    previous_end_time = None
    group_id = 1

    for session in day_sessions:
        if previous_end_time is None or (session['start_time'] - previous_end_time) > timedelta(seconds=session_division_seconds) or session['complied'] == 1:
            # Start a new group
            if current_group_sessions:
                grouped_session = {
                'group_id': group_id,
                'sessions': current_group_sessions,
                'duration': sum((s['duration'] for s in current_group_sessions), timedelta(0)),
                'target_app_duration': sum((s['duration'] for s in current_group_sessions if s['app_name'] in social_media_apps), timedelta(0)),
                'action': 1 if any(s.get('action', 0) == 1 for s in current_group_sessions) else 0,
                'start_time': current_group_sessions[0]['start_time'],
                'end_time': current_group_sessions[-1]['end_time'],
                'app_ids': [s['app_name'] for s in current_group_sessions],
                'date': current_group_sessions[0]['date'],
                'total_vibrations': sum((1 for s in current_group_sessions if s.get('action', 0) == 1), 0),
                'complied_vibrations': sum((1 for s in current_group_sessions if s.get('complied', 0) == 1 and s.get('action', 0) == 1), 0),
                'updated_duration': sum(
                        ((s['updated_duration'] if s.get('complied', 0) == 1 and 'updated_duration' in s else s['duration'])
                        for s in current_group_sessions)
                    , timedelta(0) ),
                
                'total_action_taken': sum((1 for s in current_group_sessions if s.get('action_taken', False)), 0)
                    }
                grouped_sessions.append(grouped_session)
                group_id += 1
                current_group_sessions = [session]
        else:
            current_group_sessions.append(session)
        previous_end_time = session['end_time']

    # Add the last group
    if current_group_sessions:
        grouped_session = {
            'group_id': group_id,
            'sessions': current_group_sessions,
            'duration': sum((s['duration'] for s in current_group_sessions), timedelta(0)),
            'target_app_duration': sum((s['duration'] for s in current_group_sessions if s['app_name'] in social_media_apps), timedelta(0)),
            'action': 1 if any(s.get('action', 0) == 1 for s in current_group_sessions) else 0,
            'start_time': current_group_sessions[0]['start_time'],
            'end_time': current_group_sessions[-1]['end_time'],
            'app_ids': [s['app_name'] for s in current_group_sessions],
            'date': current_group_sessions[0]['date'],
            'total_vibrations': sum((1 for s in current_group_sessions if s.get('action', 0) == 1), 0),
            'complied_vibrations': sum((1 for s in current_group_sessions if s.get('complied', 0) == 1 and s.get('action', 0) == 1), 0),
            'updated_duration': sum(
                        ((s['updated_duration'] if s.get('complied', 0) == 1 and 'updated_duration' and s.get('action', 0) == 1 in s else s['duration'])
                        for s in current_group_sessions)
                    , timedelta(0) ),
            'total_action_taken': sum((1 for s in current_group_sessions if s.get('action_taken', False)), 0)
                    
        }
        grouped_sessions.append(grouped_session)

    return grouped_sessions

def simulate(all_sessions, worker_id, compliance_rate, writer):
   
    median_target_app_usage = calculate_median_target_app_usage(all_sessions[0:7])
    short_session_length = calculate_short_session_length(all_sessions[0:7])
    query_interval_seconds = calculate_75_percentile_length(all_sessions[0:7]).total_seconds()
    # query_interval_seconds = 5 * 60 if test_type == 'C' or test_type == 'D' else short_session_length.total_seconds()
    # query_interval_seconds = short_session_length.total_seconds()
    
    
    # print(f"Median target app usage in baseline week: {median_target_app_usage}, Short session length: {short_session_length}")
    intervention_sessions = all_sessions[7:]
    agent = QLearningAgent()
    for i in range(len(intervention_sessions)):
        #simulating a day
        
        day_sessions = intervention_sessions[i]
        action_taken = 1
        # prediction loop
        for j in range(len(day_sessions)):
            session = day_sessions[j]
            session['target_group_time'] +=  session['duration'] if session['app_name'] in social_media_apps else timedelta(0)
            session['action_taken'] = False
            session['action'] = 0
            if session['app_name'] in social_media_apps and session['group_time'].total_seconds() > action_taken * query_interval_seconds:
                action_taken += 1
                session['action'] = agent.act((is_target_app(get_first_app_in_group(day_sessions, session['group_id'])), 
                                                1 if action_taken == 1 else 0, get_day_quarter(session['start_time']), 
                                                get_day_of_week(session['start_time'])))
                session['action_taken'] = True
            session['complied'] = 0
            if session['action'] == 1:
                complied = random.random() < compliance_rate
                if complied:
                    session['complied'] = 1
                    #create new group ids for all sessions after this one
                    for k in range (j, len(day_sessions)):
                        day_sessions[k]['group_id'] += 1
                    #randomly decide a time between start_time and end_time to set as end_time
                    session['updated_end_time'] = session['start_time'] + timedelta(seconds=random.randint(0, int(session['duration'].total_seconds())))
                    session['updated_duration'] = session['updated_end_time'] - session['start_time']
               

            
        
        # learning loop  
        grouped_day_sessions = get_grouped_day_sessions(day_sessions)
        # print(f"Day {i} grouped sessions: {len(grouped_day_sessions)}")
        for j in range(len(grouped_day_sessions)):
            session = grouped_day_sessions[j]
            next_session = grouped_day_sessions[j+1] if j+1 < len(grouped_day_sessions) else None
            # print(session)
            # print(f"Day {session['date']} Session {session["group_id"]} of {session["duration"].total_seconds()} seconds : , Action: {"VIBRATE" if session['action'] == 1 else "NO VIBRATE"}")
            reward = reward_fn_vibration(session['action'], session['end_time'], next_session['start_time'] if next_session else None, median_target_app_usage, session, config)
            session['reward'] = reward
            writer.writerow([
                worker_id,
                    session['group_id'],
                    session['date'],
                    session['app_ids'],
                    session['start_time'],
                    session['end_time'],
                    session['duration'].total_seconds(),
                    "Yes" if session['duration'] > short_session_length else "No",
                    session['target_app_duration'].total_seconds(),
                    "Yes" if session['action'] == 1 else "No",
                    reward,
                    session['total_vibrations'],
                    session['total_action_taken'],
                    session['complied_vibrations'],
                    session['updated_duration'].total_seconds(),
                    query_interval_seconds
                ])
            if next_session is not None:
                current_state = extract_state(get_first_app_in_group(day_sessions, session['group_id']), session)
                next_state = extract_state(get_first_app_in_group(day_sessions, next_session['group_id']), next_session)
                agent.learn(
                    tuple(current_state),
                    tuple(next_state),
                    session["action"] or 0,
                    reward
                )
                
            
            
#update group_id field in session, grouping by time gaps less than 2 minutes
def add_group_ids(day_sessions):
    group_id = 0
    group_time = timedelta(seconds=0)
    previous_end_time = None
    for session in day_sessions:
        if previous_end_time is None or (session['start_time'] - previous_end_time) > timedelta(seconds = session_division_seconds):
            group_id += 1
            group_time = timedelta(seconds=0)
        group_time += session['duration']
        session['group_time'] = group_time
        session['group_id'] = group_id
        previous_end_time = session['end_time']
    return day_sessions

def parse_time(text):
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M:%S.%f'):
        try:

            return pd.to_datetime(text,format= fmt)
        except ValueError:
            print('f')
            pass
    raise ValueError('no ' + text)

# import sim_effective_habituator_.csv and run simulate function
# import sim_effective_habituator_.csv and run simulate function
if __name__ == "__main__":
    simulations = [
        ('sim_Delayed_effect', 0.6),
        ('sim_effective_habituator_', 0.4),
        ('sim_ideal_user', 0.8),
        ('sim_non_responder', 0.05),
        ('sim_Uses_more', 0.3)
    ]
    
    for file_name, compliance_rate in simulations:
        print(f"Running simulation for {file_name} with compliance rate {compliance_rate}")
        
        original_df = pd.read_csv(f"datasets/{file_name}.csv")
        all_workers = original_df['worker_id'].unique()

        
        with open(f"outputs/{test_type}/simulation_results_{file_name}_compliance_rate_{compliance_rate}.csv", mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([
                'worker_id',
                'group_id',
                'date',
                'app_ids',
                'start_time',
                'end_time',
                'duration_seconds',
                'is_long_session',
                'target_app_duration_seconds',
                'vibration',
                'reward',
                'total_vibrations',
                'total_queries',
                'complied_vibrations',
                'updated_duration_seconds',
                'query_interval_seconds'
            ])
            
            for worker_id in all_workers:
                #filter df by worker_id
                all_sessions = []
                
                df = original_df[original_df['worker_id'] == worker_id]
                date1 = pd.to_datetime(df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S.%f')
                date2 = pd.to_datetime(df['timestamp'], errors='coerce', format='%Y-%m-%d %H:%M:%S')
                df['start_time'] = date1.fillna(date2)
                df['duration'] = df['time_seconds'].astype(float).astype(int)
                df['end_time'] = df['start_time'] + pd.to_timedelta(df['duration'], unit='s')
                for day, day_df in df.groupby('date_only'):
                    day_sessions = []
                    for _, row in day_df.iterrows():
                        session = {
                            'app_name': row['app_id'],
                            'start_time': row['start_time'],
                            'end_time': row['end_time'],
                            'duration': timedelta(seconds=row['duration']),
                            'group_id': None,
                            'group_time': timedelta(0),
                            'target_group_time': timedelta(0),
                            'date': row['date_only'],
                            'action': None
                        }
                        day_sessions.append(session)
                    all_sessions.append(add_group_ids(day_sessions))
                simulate(all_sessions, worker_id, compliance_rate, writer)
        
        print(f"Completed simulation for {file_name}")
      

/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:283: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['start_time'] = date1.fillna(date2)
/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:284: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df['time_seconds'].astype(float).astype(int)
/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:285: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

Completed simulation for sim_effective_habituator_
Running simulation for sim_ideal_user with compliance rate 0.8


/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:283: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['start_time'] = date1.fillna(date2)
/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:284: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df['time_seconds'].astype(float).astype(int)
/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:285: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

Completed simulation for sim_ideal_user
Running simulation for sim_non_responder with compliance rate 0.05


/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:283: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['start_time'] = date1.fillna(date2)
/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:284: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df['time_seconds'].astype(float).astype(int)
/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:285: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

Completed simulation for sim_non_responder
Running simulation for sim_Uses_more with compliance rate 0.3


/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:283: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['start_time'] = date1.fillna(date2)
/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:284: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df['time_seconds'].astype(float).astype(int)
/var/folders/h8/zj0wwk991mz87wf77n_lt61h0000gn/T/ipykernel_31276/1925387453.py:285: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

Completed simulation for sim_Uses_more
